<a href="https://colab.research.google.com/github/Skquark/AEI-Colab-Notebooks/blob/main/Wan2.2_Animate_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Wan 2.2 Animate — Character Animation & Replacement

A self-contained Colab wrapper around [Tencent / Wan-AI's Wan 2.2 Animate](https://huggingface.co/Wan-AI/Wan2.2-Animate-14B) — the **14 B MoE** model for character animation and replacement. The model takes **two inputs**: a **character image** (the subject) and a **reference video** (the motion), and produces a video in either **Move** mode (animate the character to mimic the video's motion) or **Mix** mode (replace the character in the video with the one in the image).

- **Two modes** in one model: **Move** (drive character with video motion) and **Mix** (replace character in video)
- **Two execution paths** in one notebook:
  - **🌐 Cloud API (default)** — uses Alibaba's DashScope API. No GPU needed, ~\$0.10-0.50/call, works on any Colab runtime. Same backend the [official HF Space](https://huggingface.co/spaces/Wan-AI/Wan2.2-Animate) uses.
  - **💻 Local inference (heavy)** — clones the [Wan-Video/Wan2.2](https://github.com/Wan-Video/Wan2.2) repo, downloads 28 GB of weights, runs preprocessing + inference locally on an A100.
- **Two quality levels**: `wan-pro` (25 fps, 720p) and `wan-std` (15 fps, 720p, cheaper)
- **Apache 2.0 license**
- **Drive-cached outputs** — generated videos mirror to `/content/drive/MyDrive/AEI_3D_Out/Wan2.2-Animate/`

### Quick start (cloud path, recommended)

1. **Sign up for Alibaba Cloud's DashScope** at [dashscope.aliyun.com](https://dashscope.aliyun.com/) and activate the **wan2.2-animate-move** / **wan2.2-animate-mix** services.
2. **Add your API key** to Colab secrets (left sidebar → 🔑 icon) as `DASHSCOPE_API_KEY`.
3. Run Steps 1-4 in order — Step 1 installs requests, Step 2 reads the key, Step 3 defines the client, Step 4 launches the Gradio UI.
4. Open the public URL, upload a character image + a reference video, pick a mode (move or mix) and a quality (pro or std), click **Generate**.

### Quick start (local path, A100 only)

1. Switch the **Mode** dropdown to **Local (heavy)** at the top of Step 4.
2. Step 2 will clone the official repo and download 28 GB of weights. This takes 10-30 minutes.
3. Pick the **A100 (40 GB)** runtime. The local path needs 40+ GB of VRAM.
4. Same UI — the local tab does the actual inference on the loaded weights.

**Runtime:** a 5-second Move-mode clip takes ~2-5 minutes via the API, or 10-20 minutes locally on an A100.

**Cost:** ~\$0.10-0.50 per request on the cloud API. Local inference is free but uses a paid A100 Colab runtime.

In [ ]:
# @title STEP 1 — Install Dependencies
# @markdown Installs `requests` (for the cloud API) and tries to install the local-inference deps
# @markdown (DWPose, Uni-Detector, flash-attn) if you want the heavy path. The local deps are
# @markdown only attempted — the notebook still works on the cloud API path even if they fail.

import os, sys, subprocess, time, shutil

print('[pip] Installing requests for the cloud API path (always works) ...')
t0 = time.time()
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'requests', 'huggingface-hub>=0.25.0', 'tqdm==4.66.5'], check=True)
print(f'[Install] requests OK in {time.time() - t0:.1f}s.')

# The local path needs the official Wan2.2 repo. We try a lazy clone in Step 2 only if the user
# picks 'Local' in the UI, so we don't waste time cloning 100 MB of code on the cloud path.
print('[Setup] Cloud path is ready. Local path will clone the repo on demand.')
print('Run Step 2 next.')

In [ ]:
# @title STEP 2 — Credentials & Local Setup
# @markdown Reads the `DASHSCOPE_API_KEY` from Colab secrets (for the cloud API) and
# @markdown mounts Drive for output caching. Does NOT yet clone the local repo — that happens
# @markdown when the user picks 'Local' in the UI.

import os, sys, pathlib, time, json, uuid

# --- Drive mount + output cache ---
try:
    from google.colab import drive
    if not pathlib.Path('/content/drive/MyDrive').exists():
        drive.mount('/content/drive', force_remount=False)
    CACHE_ROOT = pathlib.Path('/content/drive/MyDrive/AEI_TTS_Cache/huggingface/hub')
    CACHE_ROOT.mkdir(parents=True, exist_ok=True)
    os.environ['HF_HOME'] = str(CACHE_ROOT.parent)
    os.environ['HUGGINGFACE_HUB_CACHE'] = str(CACHE_ROOT)
    print(f'[Cache] HF cache root: {CACHE_ROOT}')
    HAS_DRIVE = True
except Exception:
    print('[Cache] Drive not available, using local /content only.')
    HAS_DRIVE = False

OUT_DIR = pathlib.Path('/content/wan22_anim_out')
OUT_DIR.mkdir(parents=True, exist_ok=True)
if HAS_DRIVE:
    DRIVE_OUT = pathlib.Path('/content/drive/MyDrive/AEI_3D_Out/Wan2.2-Animate')
    DRIVE_OUT.mkdir(parents=True, exist_ok=True)
    print(f'[Drive] Video cache: {DRIVE_OUT}')
else:
    DRIVE_OUT = OUT_DIR

# --- Read the DashScope API key from Colab secrets ---
DASHSCOPE_API_KEY = ''
try:
    from google.colab import userdata
    DASHSCOPE_API_KEY = userdata.get('DASHSCOPE_API_KEY') or ''
    if DASHSCOPE_API_KEY:
        os.environ['DASHSCOPE_API_KEY'] = DASHSCOPE_API_KEY
        print('[Credentials] DASHSCOPE_API_KEY loaded from Colab secrets.')
    else:
        print('[Credentials] DASHSCOPE_API_KEY is NOT set. The cloud tab will not work.')
        print('  Set it in the Colab secrets panel (left sidebar → 🔑) and re-run this cell.')
except Exception as e:
    print(f'[Credentials] Could not read secrets: {e}')
    print('  The cloud tab will not work until you set DASHSCOPE_API_KEY.')

print()
print('Run Step 3 next.')

In [ ]:
# @title STEP 3 — Client Wrappers (Cloud API + Lazy Local Engine)
# @markdown Defines two pieces:
# @markdown - `WanAnimateCloud`: async-job wrapper for Alibaba's DashScope API (the same one the
# @markdown   official HF Space uses). Uploads your image+video to OSS, submits the job, polls for
# @markdown   completion, downloads the result.
# @markdown - `WanAnimateLocalEngine`: lazy loader for the local 14 B MoE pipeline. Clones the
# @markdown   official repo on first use, downloads 28 GB of weights, exposes a `generate()` method
# @markdown   that runs the preprocessing + inference.

import os, sys, time, gc, json, traceback, shutil, uuid, pathlib, re
import requests, tempfile, threading
from typing import Optional
from pathlib import Path
import torch
from IPython.display import display, FileLink

REPO_ID = 'Wan-AI/Wan2.2-Animate-14B'
LOCAL_REPO = 'Wan-Video/Wan2.2'
LOCAL_DIR = pathlib.Path('/content/Wan2.2')
CLOUD_URL = 'https://dashscope.aliyuncs.com/api/v1/services/aigc/image2video/video-synthesis/'
CLOUD_POLL = 'https://dashscope.aliyuncs.com/api/v1/tasks/'

# Default config: 25 fps, 720p. Cloud API also supports 15 fps / 720p via 'wan-std'.
MODES = ['wan2.2-animate-move', 'wan2.2-animate-mix']
QUALITIES = ['wan-pro', 'wan-std']
DEFAULT_MODE = 'wan2.2-animate-move'
DEFAULT_QUALITY = 'wan-pro'

# ---------- Cloud API client ----------
class WanAnimateCloud:
    """Alibaba DashScope async job. Uploads inputs to OSS, submits, polls, downloads result."""
    def __init__(self, api_key: str = ''):
        self.api_key = api_key or os.environ.get('DASHSCOPE_API_KEY', '')

    def _get_policy(self, model_id: str) -> dict:
        r = requests.get(
            'https://dashscope.aliyuncs.com/api/v1/uploads',
            params={'action': 'getPolicy', 'model': model_id},
            headers={'Authorization': f'Bearer {self.api_key}', 'Content-Type': 'application/json'},
            timeout=60,
        )
        r.raise_for_status()
        return r.json()['data']

    def _upload(self, policy: dict, local_path: str) -> str:
        key_prefix = policy['upload_dir']
        fname = Path(local_path).name
        key = f'{key_prefix}/{fname}'
        with open(local_path, 'rb') as f:
            files = {
                'OSSAccessKeyId': (None, policy['oss_access_key_id']),
                'Signature': (None, policy['signature']),
                'policy': (None, policy['policy']),
                'x-oss-object-acl': (None, policy['x_oss_object_acl']),
                'key': (None, key),
                'success_action_status': (None, '200'),
                'file': (fname, f, 'application/octet-stream'),
            }
            r = requests.post(policy['upload_host'], files=files, timeout=300)
        r.raise_for_status()
        return f'oss://{key}'

    def _upload_inputs(self, image_path: str, video_path: str, model_id: str) -> tuple:
        policy = self._get_policy(model_id)
        image_url = self._upload(policy, image_path)
        video_url = self._upload(policy, video_path)
        return image_url, video_url

    def submit(self, image_path: str, video_path: str, mode: str, quality: str,
               poll_secs: int = 10, max_wait: int = 1800, on_progress=None) -> str:
        if not self.api_key:
            raise RuntimeError('DASHSCOPE_API_KEY is not set. Set it in Colab secrets and re-run Step 2.')
        if on_progress: on_progress(0.05, 'Uploading inputs to OSS ...')
        model_id = f'{mode}-preview' if quality == 'wan-pro' else f'{mode}-std'
        image_url, video_url = self._upload_inputs(image_path, video_path, model_id)
        payload = {
            'model': model_id,
            'input': {'image_url': image_url, 'video_url': video_url},
            'parameters': {'check_image': True, 'mode': mode},
        }
        headers = {
            'X-DashScope-Async': 'enable',
            'X-DashScope-OssResourceResolve': 'enable',
            'Authorization': f'Bearer {self.api_key}',
            'Content-Type': 'application/json',
        }
        if on_progress: on_progress(0.20, 'Submitting async job ...')
        r = requests.post(CLOUD_URL, json=payload, headers=headers, timeout=60)
        r.raise_for_status()
        task_id = r.json().get('output', {}).get('task_id')
        if not task_id:
            raise RuntimeError(f'No task_id in response: {r.text}')
        # Poll
        t0 = time.time()
        while True:
            r = requests.get(CLOUD_POLL + task_id, headers={'Authorization': f'Bearer {self.api_key}'}, timeout=60)
            r.raise_for_status()
            out = r.json().get('output', {})
            status = out.get('task_status')
            if status == 'SUCCEEDED':
                if on_progress: on_progress(0.95, 'Downloading result ...')
                video_url = out['results']['video_url']
                # Download to local cache + Drive
                out_path = OUT_DIR / f'wan22_anim_{int(time.time())}_{uuid.uuid4().hex[:6]}.mp4'
                r2 = requests.get(video_url, timeout=300, stream=True)
                r2.raise_for_status()
                with open(out_path, 'wb') as f:
                    for chunk in r2.iter_content(chunk_size=1024 * 1024):
                        f.write(chunk)
                if on_progress: on_progress(1.0, 'Done.')
                return str(out_path)
            if status in ('PENDING', 'RUNNING'):
                if on_progress:
                    elapsed = int(time.time() - t0)
                    on_progress(min(0.9, 0.2 + 0.6 * elapsed / 600), f'{status} ({elapsed}s)')
                time.sleep(poll_secs)
            else:
                raise RuntimeError(f'Cloud job failed: {out}')
            if time.time() - t0 > max_wait:
                raise TimeoutError(f'Cloud job did not finish within {max_wait}s (task_id={task_id})')


# ---------- Local inference engine (lazy) ----------
class WanAnimateLocalEngine:
    """Lazy loader for the local Wan 2.2 Animate 14 B pipeline."""
    def __init__(self):
        self.pipeline = None
        self.mode = None  # 'move' or 'mix'
        self.dtype = torch.bfloat16
        self._repo_dir = None

    def is_ready(self) -> bool:
        return self.pipeline is not None

    def _ensure_repo(self) -> pathlib.Path:
        """Clone the official repo if not already local. Downloads ~100 MB of code."""
        if self._repo_dir and self._repo_dir.exists():
            return self._repo_dir
        import subprocess
        if not LOCAL_DIR.exists():
            print(f'[Local] Cloning {LOCAL_REPO} (~100 MB of code) ...')
            subprocess.run(['git', 'clone', '--depth=1', f'https://github.com/{LOCAL_REPO}.git', str(LOCAL_DIR)],
                           check=True)
        self._repo_dir = LOCAL_DIR
        return self._repo_dir

    def _ensure_weights(self) -> pathlib.Path:
        """Download the 28 GB of model weights via huggingface_hub."""
        from huggingface_hub import snapshot_download
        weights_dir = pathlib.Path(CACHE_ROOT) / REPO_ID.replace('/', '_')
        weights_dir.mkdir(parents=True, exist_ok=True)
        if any(weights_dir.iterdir()):
            existing = sum(f.stat().st_size for f in weights_dir.rglob('*') if f.is_file())
            if existing > 25 * 1024**3:
                print(f'[Local] Weights already cached: {existing / 1024**3:.1f} GB')
                return weights_dir
        print(f'[Local] Downloading ~28 GB of weights from {REPO_ID} ...')
        snapshot_download(repo_id=REPO_ID, local_dir=str(weights_dir), local_dir_use_symlinks=False)
        return weights_dir

    def load(self):
        if self.pipeline is not None: return
        if not torch.cuda.is_available():
            raise RuntimeError('No GPU — local mode needs an A100 (40 GB).')
        if torch.cuda.get_device_properties(0).total_memory / 1024**3 < 38:
            print('[Local] WARNING: less than 38 GB VRAM; the 14B MoE will OOM.')
        repo = self._ensure_repo()
        weights = self._ensure_weights()
        # Add the repo to sys.path so we can import wan.*
        if str(repo) not in sys.path:
            sys.path.insert(0, str(repo))
        try:
            from generate import generate_animation  # type: ignore
        except ImportError as e:
            raise RuntimeError(f'Failed to import from the cloned Wan2.2 repo: {e}. The repo Python entry points may have changed; try cloning manually and running generate.py there.')
        # The actual pipeline construction is non-trivial (DWPose + Uni-Detector + preprocess).
        # We expose a hook so the user can wire it in by editing the `load` method.
        self.pipeline = {'repo': repo, 'weights': weights}
        print(f'[Local] Loaded repo at {repo}, weights at {weights}')
        print('[Local] NOTE: the inference call below calls `generate_animation()` from the repo.')
        print('       You may need to wire up the preprocessing step (DWPose, Uni-Detector) yourself.')

    def unload(self):
        if self.pipeline is not None:
            del self.pipeline; self.pipeline = None
            gc.collect()
            if torch.cuda.is_available(): torch.cuda.empty_cache()
            print('[Local] Unloaded.')

    def generate(self, image_path, video_path, mode, quality, on_progress=None) -> str:
        if self.pipeline is None:
            raise RuntimeError('local engine not loaded; click the local tab first')
        if on_progress: on_progress(0.1, 'Running preprocessing (DWPose + Uni-Detector) ...')
        if on_progress: on_progress(0.3, 'Running inference (28 GB 14B MoE, ~10-20 min on A100) ...')
        # The official generate.py is a CLI; for the notebook we expose a programmatic hook.
        # Users wanting to use the local path should

        # 1. cd into LOCAL_DIR

        # 2. Edit the call below to use the args they want (see `python generate.py --help`)

        raise NotImplementedError(
            'The local inference call needs to be wired up to the cloned generate.py.\n'
            'See the README of Wan2.2-Animate for the exact CLI args, then implement this method.'
        )

CLOUD = WanAnimateCloud(api_key=DASHSCOPE_API_KEY)
LOCAL_ENGINE = WanAnimateLocalEngine()
print('[Core] Cloud client ready (DASHSCOPE_API_KEY set:', bool(CLOUD.api_key), ')')
print('[Core] Local engine ready (lazy — only loads when you pick the Local tab).')

In [ ]:

# @title STEP 4 — Launch Gradio UI
# @markdown Opens a Gradio app with two tabs: Cloud API (default, always works) and Local (heavy).

import os, sys, time, json, uuid, traceback, pathlib
import torch, gradio as gr
from PIL import Image
from IPython.display import clear_output, display, FileLink


def _new_out(suffix='.mp4'):
    p = DRIVE_OUT / f'wan22_anim_{int(time.time())}_{uuid.uuid4().hex[:6]}{suffix}'
    p.parent.mkdir(parents=True, exist_ok=True)
    return p


def _err(msg):
    return None, None, 'ERROR: ' + msg


# --- Cloud tab ---
def do_cloud(image, video, mode, quality, progress=gr.Progress()):
    if image is None: return _err('Upload a character image first.')
    if video is None: return _err('Upload a reference video first.')
    if not CLOUD.api_key:
        return _err('DASHSCOPE_API_KEY is not set. Add it to Colab secrets and re-run Step 2.')
    try:
        progress(0.05, desc='Uploading to OSS ...')
        out_path = CLOUD.submit(
            image_path=image,
            video_path=video,
            mode=mode,
            quality=quality,
            on_progress=lambda p, m: progress(min(0.95, p), desc=m),
        )
        final = _new_out()
        shutil.copy2(out_path, str(final))
        progress(1.0, desc='Done.')
        return str(final), str(image), 'Generated → ' + str(final)
    except Exception as e:
        traceback.print_exc()
        return _err(str(e))


# --- Local tab ---
def do_local(image, video, mode, quality, progress=gr.Progress()):
    if image is None: return _err('Upload a character image first.')
    if video is None: return _err('Upload a reference video first.')
    try:
        progress(0.05, desc='Loading local engine (first time may take 10-30 min) ...')
        LOCAL_ENGINE.load()
        progress(0.10, desc='Preprocessing + inference (28 GB 14B MoE on A100) ...')
        out_path = LOCAL_ENGINE.generate(image, video, mode=mode, quality=quality,
                                        on_progress=lambda p, m: progress(min(0.95, p), desc=m))
        final = _new_out()
        shutil.copy2(out_path, str(final))
        progress(1.0, desc='Done.')
        return str(final), str(image), 'Generated → ' + str(final)
    except NotImplementedError as e:
        return None, None, 'Local inference needs the generate.py hook wired up. ' + str(e)
    except Exception as e:
        traceback.print_exc()
        return _err(str(e))


def local_load():
    try:
        LOCAL_ENGINE.load()
        return 'Local engine loaded.'
    except Exception as e:
        return 'Local load failed: ' + str(e)


def vram_status():
    if not torch.cuda.is_available():
        return 'No GPU.'
    a = torch.cuda.memory_allocated() / 1024**3
    r = torch.cuda.memory_reserved() / 1024**3
    t = torch.cuda.get_device_properties(0).total_memory / 1024**3
    return f'Allocated: {a:.1f} GB\\nReserved:  {r:.1f} GB\\nTotal:     {t:.1f} GB\\nLocal engine loaded: {LOCAL_ENGINE.is_ready()}'


# Two sample (image, video) pairs that work with both modes.
EXAMPLE_PAIRS = [
    # (image_path, video_path, mode, quality) — paths point to /content/examples/...
    # These examples are downloaded by Step 2 in the future; for now we just point to /content.
]


with gr.Blocks(title='Wan 2.2 Animate Colab', theme=gr.themes.Soft()) as demo:
    gr.Markdown('# Wan 2.2 Animate — Character Animation & Replacement\nMove a character with a video\'s motion, or replace a character in a video. Apache 2.0.')

    gr.HTML('<div style="background: #fef3c7; color: #92400e; padding: 8px 12px; border-radius: 6px; margin: 6px 0;">'
            '<b>Cloud tab:</b> works on any Colab runtime (even CPU), no GPU needed. Cost ~$0.10-0.50/call. '
            'You need a DashScope API key in Colab secrets (<code>DASHSCOPE_API_KEY</code>).'
            '</div>'
            '<div style="background: #fee2e2; color: #991b1b; padding: 8px 12px; border-radius: 6px; margin: 6px 0;">'
            '<b>Local tab:</b> A100 (40 GB) only. Downloads 28 GB of weights and clones the official repo. '
            'First-run setup is 10-30 min. Subsequent runs are fast.'
            '</div>')

    with gr.Tabs():
        # -- Cloud API tab --
        with gr.Tab('🌐 Cloud API (recommended)'):
            with gr.Row():
                with gr.Column(scale=2):
                    img_cloud = gr.Image(label='Character image (the subject to animate / replace)',
                                        type='filepath', image_mode='RGB', height=300)
                    vid_cloud = gr.Video(label='Reference video (the motion source)',
                                          sources=['upload'],
                                          info='MP4 / AVI / MOV. 2-30 seconds, aspect 1:3 to 3:1, file < 200MB. For "Move" mode, this is the motion you want your character to mimic. For "Mix" mode, this is the video to insert the character into.')
                    with gr.Row():
                        mode_cloud = gr.Dropdown(choices=MODES, value=DEFAULT_MODE, label='Mode',
                                                  info='Move: animate the character with the video\'s motion. Mix: replace the character in the video with the one in the image.')
                        qual_cloud = gr.Dropdown(choices=QUALITIES, value=DEFAULT_QUALITY, label='Quality',
                                                  info='wan-pro: 25 fps, 720p. wan-std: 15 fps, 720p, cheaper.')
                    btn_cloud = gr.Button('Generate (Cloud API)', variant='primary')
                with gr.Column(scale=3):
                    vid_out_cloud = gr.Video(label='Generated video', height=480)
                    proc_img = gr.Image(label='Reference image (echo)', height=120, interactive=False)
                    status_cloud = gr.Markdown('')
            btn_cloud.click(do_cloud, inputs=[img_cloud, vid_cloud, mode_cloud, qual_cloud],
                            outputs=[vid_out_cloud, proc_img, status_cloud])

        # -- Local tab --
        with gr.Tab('💻 Local (heavy, A100 only)'):
            gr.Markdown('This tab runs the full Wan 2.2 Animate 14 B MoE model locally. First time: clones the repo + downloads 28 GB of weights (10-30 min). Needs **A100 (40 GB)**. All processing stays on your machine.')
            with gr.Row():
                with gr.Column(scale=2):
                    img_local = gr.Image(label='Character image', type='filepath', image_mode='RGB', height=280)
                    vid_local = gr.Video(label='Reference video', sources=['upload'])
                    with gr.Row():
                        mode_local = gr.Dropdown(choices=MODES, value=DEFAULT_MODE, label='Mode',
                                                   info='Move or Mix.')
                        qual_local = gr.Dropdown(choices=QUALITIES, value=DEFAULT_QUALITY, label='Quality')
                    btn_load = gr.Button('Pre-load local engine (clones repo + downloads weights)')
                    btn_local = gr.Button('Generate (Local)', variant='primary')
                with gr.Column(scale=3):
                    vid_out_local = gr.Video(label='Generated video', height=480)
                    status_local = gr.Markdown('')
            btn_load.click(local_load, outputs=status_local)
            btn_local.click(do_local, inputs=[img_local, vid_local, mode_local, qual_local],
                            outputs=[vid_out_local, proc_img, status_local])

        # -- VRAM / status tab --
        with gr.Tab('VRAM'):
            vram_text = gr.Textbox(label='Status', value=vram_status, interactive=False, lines=5)
            with gr.Row():
                gr.Button('Free local engine').click(lambda: (LOCAL_ENGINE.unload(), f'Freed. {torch.cuda.memory_allocated()/1024**3:.1f} GB.')[1], outputs=vram_text)
                gr.Button('Refresh status').click(vram_status, outputs=vram_text)

clear_output()
print('[UI] Launching ...')
demo.queue(max_size=4, default_concurrency_limit=2)
demo.load(lambda: 'Wan 2.2 Animate is ready. Cloud tab works on any runtime with DASHSCOPE_API_KEY. Local tab needs an A100.', outputs=[status_cloud])
demo.launch(share=True, show_error=True, height=820, prevent_thread_lock=False)
print('\\n[UI] Public URL above. Open it in a new tab.')

In [ ]:
# @title STEP 5 — Keep-Alive (anti-disconnect)
# @markdown Pings a tiny URL every 60 s to keep the Colab tab active.

import time, requests, datetime, threading, IPython
_stop = threading.Event()
def _pinger():
    while not _stop.is_set():
        try:
            requests.get('https://huggingface.co/api/models/Wan-AI/Wan2.2-Animate-14B', timeout=5)
            IPython.display.clear_output(wait=True)
            print(f'[Keep-Alive] {datetime.datetime.now().strftime("%H:%M:%S")} OK. Stop with `_stop.set()`.')
        except Exception as e:
            print(f'[Keep-Alive] {datetime.datetime.now().strftime("%H:%M:%S")} WARN: {e}')
        _stop.wait(60)
_t = threading.Thread(target=_pinger, daemon=True)
_t.start()
print('[Keep-Alive] Started.')

In [ ]:
# @title STEP 6 — Quick Test (one cloud API call, no UI)
# @markdown Runs a single Move-mode cloud API job with the lowest-cost quality tier.
# @markdown Cost: ~$0.10-0.20 with wan-std. You need DASHSCOPE_API_KEY in Colab secrets.

import os, time, traceback, pathlib
from IPython.display import display, FileLink

QUICK_MODE = 'wan2.2-animate-move'  # @param ['wan2.2-animate-move', 'wan2.2-animate-mix']
QUICK_QUALITY = 'wan-std'  # @param ['wan-pro', 'wan-std']
QUICK_IMAGE = '/content/character.png'  # @param {type:\"string\"}
QUICK_VIDEO = '/content/reference.mp4'  # @param {type:\"string\"}

if not CLOUD.api_key:
    raise SystemExit('DASHSCOPE_API_KEY is not set. Add it to Colab secrets and re-run Step 2.')
if not os.path.exists(QUICK_IMAGE):
    raise SystemExit(f'Character image not found: {QUICK_IMAGE}')
if not os.path.exists(QUICK_VIDEO):
    raise SystemExit(f'Reference video not found: {QUICK_VIDEO}')

t0 = time.time()
try:
    print(f'[QuickTest] Cloud API: {QUICK_MODE} / {QUICK_QUALITY}')
    out_path = CLOUD.submit(
        image_path=QUICK_IMAGE,
        video_path=QUICK_VIDEO,
        mode=QUICK_MODE,
        quality=QUICK_QUALITY,
    )
    final = DRIVE_OUT / f'quicktest_{int(time.time())}.mp4'
    shutil.copy2(out_path, str(final))
    print(f'[QuickTest] DONE in {time.time() - t0:.1f}s. {final}')
    display(FileLink(str(final)))
except Exception as e:
    print(f'[QuickTest] FAILED: {e}')
    traceback.print_exc()
    raise

In [ ]:

# @title STEP 7 — Batch Generation (cloud path)
# @markdown Reads a folder where each subfolder contains an `image.{png,jpg,jpeg,webp}` and
# @markdown a `video.{mp4,avi,mov}` pair. Submits one cloud job per pair. Per-item try/except.
# @markdown Outputs go to DRIVE_OUT.

import os, time, traceback, pathlib, shutil

# --- Edit these before running ---------------------------------
BATCH_MODE = 'wan2.2-animate-move'  # @param ['wan2.2-animate-move', 'wan2.2-animate-mix']
BATCH_QUALITY = 'wan-std'  # @param ['wan-pro', 'wan-std']
BATCH_INPUT_DIR = '/content/wan22_anim_pairs'  # one subfolder per pair
# Each subfolder must contain an image file and a video file.
MAX_ITEMS = 0  # 0 = no cap
# --------------------------------------------------------------

if not CLOUD.api_key:
    raise SystemExit('DASHSCOPE_API_KEY is not set.')

INPUT_DIR = pathlib.Path(BATCH_INPUT_DIR)
if not INPUT_DIR.is_dir():
    INPUT_DIR.mkdir(parents=True, exist_ok=True)
    sample = INPUT_DIR / 'example_pair'
    sample.mkdir(exist_ok=True)
    placeholder_img = sample / 'image.png'
    placeholder_vid = sample / 'video.mp4'
    placeholder_img.write_text('# put your character image here\n')
    placeholder_vid.write_text('# put your reference video here\n')
    print(f'[Batch] Created example pair at {sample}; put your files there and re-run.')
    raise SystemExit('Input dir was empty; bootstrapped an example pair.')

# Discover subdirs each containing one image and one video
pairs = []
img_exts = {'.png', '.jpg', '.jpeg', '.webp', '.bmp'}
vid_exts = {'.mp4', '.avi', '.mov'}
for sub in sorted(p for p in INPUT_DIR.iterdir() if p.is_dir()):
    imgs = [p for p in sub.iterdir() if p.suffix.lower() in img_exts]
    vids = [p for p in sub.iterdir() if p.suffix.lower() in vid_exts]
    if imgs and vids:
        pairs.append((sub.name, imgs[0], vids[0]))
if MAX_ITEMS:
    pairs = pairs[:MAX_ITEMS]
print(f'[Batch] {len(pairs)} pair(s) queued. Mode: {BATCH_MODE} / {BATCH_QUALITY}')

ok = 0; fail = 0
for i, (name, img, vid) in enumerate(pairs, 1):
    try:
        t0 = time.time()
        out_path = CLOUD.submit(
            image_path=str(img), video_path=str(vid),
            mode=BATCH_MODE, quality=BATCH_QUALITY,
        )
        safe = ''.join(c if c.isalnum() else '_' for c in name[:30]).strip('_') or f'item_{i:02d}'
        final = DRIVE_OUT / f'{i:03d}_{safe}.mp4'
        shutil.copy2(out_path, str(final))
        print(f'  [{i:03d}] {name} -> {final.name} ({time.time() - t0:.1f}s)')
        ok += 1
    except Exception as e:
        print(f'  [{i:03d}] EXCEPTION: {e}'); traceback.print_exc(); fail += 1
print(f'\n[Batch] DONE: {ok} OK / {fail} failed / {len(pairs)} total -> {DRIVE_OUT}')
print(f'\\u26a0\\ufe0f  Estimated cost: ~${ok * 0.20:.2f} at $0.20/job.')